All Dependencies

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.applications import VGG19
from sklearn.model_selection import train_test_split
from glob import glob

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Mount Drive (Execute only if using Colab)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Load Dataset

In [ ]:
file_path = "/content/drive/MyDrive/Tasks/SR_1/Dataset"

# Initialise parameters
BATCH_SIZE = 16
PRETRAIN_EPOCHS = 50
GAN_EPOCHS = 100
SEED = 42
LAMBDA_PIXEL = 1.0
LAMBDA_VGG = 0.006
LAMBDA_ADV = 0.001
np.random.seed(SEED)
tf.random.set_seed(SEED)
CLASS_NAMES = ["LR","HR"]

def load_dataset(dataset_dir):

  img_dict = defaultdict(list)
  LR = []
  HR = []

  for d in os.listdir(dataset_dir):
    if d == "LR":
      lr_dir = os.path.join(dataset_dir, d)
      for img in os.listdir(lr_dir):
        if img.endswith(".npy"):
          img_path = os.path.join(lr_dir, img)
          img_data = np.load(img_path)
          img_data = np.transpose(img_data, (1, 2, 0))
          img_dict[img].append(img_data)
    if d == "HR":
      hr_dir = os.path.join(dataset_dir, d)
      for img in os.listdir(hr_dir):
        if img.endswith(".npy"):
          img_path = os.path.join(hr_dir, img)
          img_data = np.load(img_path)
          img_data = np.transpose(img_data, (1, 2, 0))
          img_dict[img].append(img_data)


  for img_set in img_dict.values():
    LR.append(img_set[0])
    HR.append(img_set[1])

  LR = np.array(LR)
  HR = np.array(HR)

  #90-10 split
  hr_train,hr_test,lr_train,lr_test = train_test_split(HR,LR,test_size=0.10,random_state=SEED)
  return hr_train,hr_test,lr_train,lr_test

# Convert to RGB img
def to_3ch(a):
    if a.shape[-1] == 1:
        a = np.repeat(a, 3, axis=-1)
    return a.astype(np.float32)

hr_train, hr_test, lr_train, lr_test = load_dataset(file_path)
hr_train = to_3ch(hr_train)
hr_test = to_3ch(hr_test)
lr_train = to_3ch(lr_train)
lr_test = to_3ch(lr_test)

fig,axes=plt.subplots(2,2,figsize=(16,8))

for i in range(2):
    axes[0,i].imshow(lr_test[i]); axes[0,i].set_title("LR")
    axes[0,i].axis('off')
    axes[1,i].imshow(hr_test[i]); axes[1,i].set_title("HR")
    axes[1,i].axis('off')

plt.suptitle("LR (top) vs HR (bottom) — Simulated Data")
plt.tight_layout()
plt.show()

Build Model

In [ ]:
### BUILD GENERATOR
def res_block(x, f=64):
    s = x
    x = layers.Conv2D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.PReLU(shared_axes=[1,2])(x)
    x = layers.Conv2D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    return layers.Add()([x, s])

# Input: 75×75×3 → Output: 150×150×3
inp = keras.Input(shape=(75, 75, 3))
x = layers.Conv2D(64, 9, padding='same')(inp)
x = layers.PReLU(shared_axes=[1,2])(x)
skip = x

for _ in range(16):
    x = res_block(x, 64)

x = layers.Conv2D(64, 3, padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Add()([x, skip])

# Single 2× upsample: 75→150 via pixel shuffle
x = layers.Conv2D(256, 3, padding='same')(x)
x = layers.Lambda(lambda t: tf.nn.depth_to_space(t, 2))(x)
x = layers.PReLU(shared_axes=[1,2])(x)

out = layers.Conv2D(3, 9, padding='same', activation='sigmoid')(x)  # 150×150×3

generator = keras.Model(inp, out, name="Generator")

### BUILD DISCRIMINATOR
def d_block(x, f, s=1, bn=True):
    x = layers.Conv2D(f, 3, strides=s, padding='same')(x)
    if bn: x = layers.BatchNormalization()(x)
    return layers.LeakyReLU(0.2)(x)

d_inp = keras.Input(shape=(150, 150, 3))
dx = d_block(d_inp, 64, 1, bn=False)
dx = d_block(dx, 64, 2)
dx = d_block(dx, 128, 1)
dx = d_block(dx, 128, 2)
dx = d_block(dx, 256, 1)
dx = d_block(dx, 256, 2)
dx = d_block(dx, 512, 1)
dx = d_block(dx, 512, 2)
dx = layers.Flatten()(dx)
dx = layers.Dense(1024)(dx)
dx = layers.LeakyReLU(0.2)(dx)
d_out = layers.Dense(1, activation='sigmoid')(dx)

discriminator = keras.Model(d_inp, d_out, name="Discriminator")

### VGG19 FOR L1 LOSS
vgg = VGG19(include_top=False, weights='imagenet', input_shape=(150, 150, 3))
vgg.trainable = False
vgg_feat = keras.Model(vgg.input, vgg.get_layer('block5_conv4').output, name="VGG_feat")


Pre-Train Model

In [ ]:
def compute_metrics(hr, sr):
    mse = np.mean((hr - sr)**2, axis=(1,2,3))
    psnr = 10 * np.log10(1.0 / (mse + 1e-10))
    ssim = tf.image.ssim(hr, sr, max_val=1.0).numpy()
    return np.mean(mse), np.mean(psnr), np.mean(ssim)

generator.compile(optimizer=optimizers.Adam(1e-4), loss='mae')
generator.fit(lr_train, hr_train, validation_data=(lr_test, hr_test),
    epochs=PRETRAIN_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)], verbose=1)

m, p, s = compute_metrics(hr_test, generator.predict(lr_test, verbose=0))
print(f"\nAfter L1 pretrain — MSE: {m:.6f} | PSNR: {p:.2f} dB | SSIM: {s:.4f}")

Train Model

In [ ]:
gen_opt = optimizers.Adam(1e-4, beta_1=0.9)
disc_opt = optimizers.Adam(1e-4, beta_1=0.9)
bce = keras.losses.BinaryCrossentropy()
mae_fn = keras.losses.MeanAbsoluteError()

@tf.function
def train_step(lr_b, hr_b):
    bs = tf.shape(lr_b)[0]
    real_l = tf.ones((bs, 1)); fake_l = tf.zeros((bs, 1))
    # Discriminator
    with tf.GradientTape() as dt:
        sr = generator(lr_b, training=True)
        d_loss = (bce(real_l, discriminator(hr_b, training=True)) +
                  bce(fake_l, discriminator(sr, training=True))) / 2
    disc_opt.apply_gradients(zip(dt.gradient(d_loss, discriminator.trainable_variables),
                                  discriminator.trainable_variables))
    # Generator
    with tf.GradientTape() as gt:
        sr = generator(lr_b, training=True)
        g_loss = (LAMBDA_PIXEL * mae_fn(hr_b, sr) +
                  LAMBDA_VGG * mae_fn(vgg_feat(hr_b), vgg_feat(sr)) +
                  LAMBDA_ADV * bce(real_l, discriminator(sr, training=True)))
    gen_opt.apply_gradients(zip(gt.gradient(g_loss, generator.trainable_variables),
                                 generator.trainable_variables))
    return g_loss, d_loss

# %%
nb = len(lr_train) // BATCH_SIZE
hist = {'g':[], 'd':[], 'psnr':[], 'ssim':[]}
best_psnr = 0

for ep in range(GAN_EPOCHS):
    idx = np.random.permutation(len(lr_train))
    eg = ed = 0
    for b in range(nb):
        s = b * BATCH_SIZE
        gl, dl = train_step(lr_train[idx[s:s+BATCH_SIZE]], hr_train[idx[s:s+BATCH_SIZE]])
        eg += gl.numpy(); ed += dl.numpy()
    eg /= nb; ed /= nb

    sr_v = generator.predict(lr_test, verbose=0)
    _, pv, sv = compute_metrics(hr_test, sr_v)
    hist['g'].append(eg); hist['d'].append(ed); hist['psnr'].append(pv); hist['ssim'].append(sv)

    if pv > best_psnr:
        best_psnr = pv
        generator.save("best_generator_simulated.keras")

    if (ep+1) % 10 == 0 or ep == 0:
        print(f"Ep {ep+1:3d}/{GAN_EPOCHS} | G:{eg:.4f} D:{ed:.4f} | PSNR:{pv:.2f} SSIM:{sv:.4f}")

Plot Model

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].plot(hist['g'], label='Gen'); ax[0].plot(hist['d'], label='Disc')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist['psnr']); ax[1].set_title('PSNR (dB)'); ax[1].grid(alpha=0.3)
ax[2].plot(hist['ssim']); ax[2].set_title('SSIM'); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("curves_VIA.png", dpi=150); plt.show()

Test Model

In [ ]:
generator = keras.models.load_model("best_generator_simulated.keras")
sr_test = generator.predict(lr_test, verbose=0)
mf, pf, sf = compute_metrics(hr_test, sr_test)

# Bicubic baseline: resize 75→150
bic = np.clip(tf.image.resize(lr_test, [150, 150], method='bicubic').numpy(), 0, 1)
mb, pb, sb = compute_metrics(hr_test, bic)

print("=" * 55)
print("FINAL METRICS — Simulated Data (75→150)")
print("=" * 55)
print(f"  {'Method':<20s} {'MSE':>10s} {'PSNR':>10s} {'SSIM':>10s}")
print(f"  {'Bicubic':<20s} {mb:>10.6f} {pb:>10.2f} {sb:>10.4f}")
print(f"  {'SRGAN (ours)':<20s} {mf:>10.6f} {pf:>10.2f} {sf:>10.4f}")

# %%
n = 5
fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
for i in range(n):
    j = np.random.randint(len(lr_test))
    lr_up = tf.image.resize(lr_test[j:j+1], [150, 150]).numpy()[0]
    axes[0,i].imshow(np.clip(lr_up,0,1)); axes[0,i].set_title("LR→150 (bicubic)"); axes[0,i].axis('off')
    axes[1,i].imshow(np.clip(sr_test[j],0,1)); axes[1,i].set_title("SR (ours)"); axes[1,i].axis('off')
    axes[2,i].imshow(hr_test[j]); axes[2,i].set_title("HR 150×150"); axes[2,i].axis('off')
plt.suptitle(f"Simulated SR (75→150) — PSNR: {pf:.2f} dB, SSIM: {sf:.4f}", fontsize=14)
plt.tight_layout(); plt.savefig("sr_results_simulated.png", dpi=150); plt.show()

generator.save("generator_simulated_final.keras")
print("Saved: generator_simulated_final.keras")

Download Simulated Generator (Execute only if using Colab)

In [ ]:
from google.colab import files
files.download("generator_simulated_final.keras")